# Guilty plea and mitigating factor sentence adjustments — percentage decrease

Calculates the percentage decreases attributable to the **guilty plea** stages and to
each common mitigating factor, using the **verified** extracted features in the shared
MongoDB cache.

## Guilty plea

For each guilty-plea entry:

```
base = pre_plea sentence   (notional minus mitigation)
pct  = plea reduction months / base
```

The plea stages in the data map to the model's plea options:
`Up to committal` -> earliest opportunity (model 33.3%), `After committal` -> before
trial dates set (25%), `After dates fixed` -> before trial starts (22.5%), `First day`
(~20%), `During trial` (15%).  District Court `Plea day` is a separate category.

## Mitigating factors

Two bases are tested for the mitigation reduction:

```
before plea : pct = reduction / notional_sentence_months
after plea  : pct = reduction / (notional_sentence_months - plea_reduction_months)
```

- **Assistance** (all four variants) is calculated on the **after-plea** base.
- The **other** factors (Self-consumption, Young offender, Rehabilitation programme) are
  calculated on **both** bases and the error rate decides which fits better.

Only charges with a measurable, non-inferred reduction are included.  Medians are
computed after removing outliers (1.5 x IQR Tukey fences per factor).


In [1]:

import sys
from pathlib import Path

import numpy as np
import pandas as pd

notebook_dir = Path.cwd().resolve()
shared_dir = notebook_dir if (notebook_dir / ".cache").exists() else notebook_dir.parent
if str(shared_dir) not in sys.path:
	sys.path.insert(0, str(shared_dir))

from linear_interpolation_model import flatten_documents, load_documents

documents, cache_metadata = load_documents(shared_dir, refresh_cache=False)
trial_rows, effect_rows = flatten_documents(documents)

charge_cols = [
	"neutral_citation",
	"trial_index",
	"charge_no",
	"defendant_id",
	"role_catalogue_key",
]

plea_effects = effect_rows[effect_rows["stage"] == "plea"].copy()
plea_effects = plea_effects.merge(
	trial_rows[charge_cols].drop_duplicates("role_catalogue_key"),
	on="role_catalogue_key",
	how="left",
)
plea_effects["pct_decrease"] = plea_effects["effect_fraction"] * 100

plea_reduction = plea_effects.groupby("role_catalogue_key")["adjustment_months"].first()

mitigation = effect_rows[effect_rows["stage"] == "mitigation"].copy()
mitigation = mitigation.merge(
	trial_rows[charge_cols].drop_duplicates("role_catalogue_key"),
	on="role_catalogue_key",
	how="left",
)
mitigation["plea_reduction_months"] = (
	mitigation["role_catalogue_key"].map(plea_reduction).fillna(0)
)
mitigation["base_after_plea"] = (
	mitigation["base_months"] - mitigation["plea_reduction_months"]
)
mitigation["pct_before"] = mitigation["adjustment_months"] / mitigation["base_months"] * 100
mitigation["pct_after"] = mitigation["adjustment_months"] / mitigation["base_after_plea"] * 100

print(f"Documents: {len(documents)}")
print(f"Trial rows: {len(trial_rows)}")
print(f"Plea effect rows: {len(plea_effects)}")
print(f"Mitigation effect rows: {len(mitigation)}")


Documents: 2308
Trial rows: 3004
Plea effect rows: 1742
Mitigation effect rows: 667


In [2]:

def factor_summary(frame, pct_col="pct_decrease"):
	rows = []
	for factor, group in frame.groupby("canonical_factor", sort=False):
		pooled = group["adjustment_months"].sum() / group["base_months"].sum() * 100
		rows.append({
			"factor": factor,
			"charges": len(group),
			"mean_pct": group[pct_col].mean(),
			"median_pct": group[pct_col].median(),
			"min_pct": group[pct_col].min(),
			"max_pct": group[pct_col].max(),
			"pooled_pct": pooled,
		})
	return pd.DataFrame(rows)


plea_map = {
	"Guilty plea: Up to committal": "earliest opportunity (model 33.3%)",
	"Guilty plea: After committal": "before trial dates set (model 25%)",
	"Guilty plea: After dates fixed": "before trial starts (model 22.5%)",
	"Guilty plea: First day": "first day of trial (model 20%)",
	"Guilty plea: During trial": "during the trial (model 15%)",
	"Guilty plea: Plea day": "plea day - district court",
	"Guilty plea: Unknown": "stage unknown",
	"Guilty plea: Other": "other",
}
plea_report = factor_summary(plea_effects)
plea_report["model_label"] = plea_report["factor"].map(plea_map)
from IPython.display import display
print("Guilty plea percentage decrease per stage (base = pre-plea sentence):")
display(
	plea_report[["factor", "model_label", "charges", "median_pct", "mean_pct", "pooled_pct"]]
	.round(2)
)
print("Model defaults: earliest 33.3%, before trial dates set 25%, before trial starts "
      "22.5%, first day 20%, during trial 15%")


Guilty plea percentage decrease per stage (base = pre-plea sentence):


,factor,model_label,charges,median_pct,mean_pct,pooled_pct
0,Guilty plea: Plea day,plea day - district court,473,33.33,33.71,33.66
1,Guilty plea: Up to committal,earliest opportunity (model 33.3%),678,33.33,33.53,33.57
2,Guilty plea: First day,first day of trial (model 20%),20,25.00,27.44,29.22
3,Guilty plea: Unknown,stage unknown,428,33.33,33.42,33.50
4,Guilty plea: During trial,during the trial (model 15%),18,21.11,22.36,20.13
5,Guilty plea: After dates fixed,before trial starts (model 22.5%),26,25.00,23.96,23.62
6,Guilty plea: Other,other,30,33.33,30.48,30.45
7,Guilty plea: After committal,before trial dates set (model 25%),69,33.33,31.55,31.23


Model defaults: earliest 33.3%, before trial dates set 25%, before trial starts 22.5%, first day 20%, during trial 15%


In [3]:

def iqr_outlier_mask(frame, value_col):
	mask = pd.Series(False, index=frame.index)
	for factor, group in frame.groupby("canonical_factor", sort=False):
		q1 = group[value_col].quantile(0.25)
		q3 = group[value_col].quantile(0.75)
		iqr = q3 - q1
		mask.loc[group.index] = (
			(group[value_col] < q1 - 1.5 * iqr) | (group[value_col] > q3 + 1.5 * iqr)
		)
	return mask


assistance_factors = [
	"Assistance - limited",
	"Assistance - useful",
	"Assistance - testify",
	"Assistance - risk",
]
other_factors = [
	"Self-consumption",
	"Young offender",
	"Rehabilitation programme",
]
all_mitigating_factors = assistance_factors + other_factors

assist = mitigation[
	mitigation["canonical_factor"].isin(assistance_factors)
	& (mitigation["base_after_plea"] > 0)
].copy()
assist_mask = iqr_outlier_mask(assist, "pct_after")
assist_clean = assist[~assist_mask]

others = mitigation[
	mitigation["canonical_factor"].isin(other_factors)
	& (mitigation["base_after_plea"] > 0)
].copy()
others_mask_before = iqr_outlier_mask(others, "pct_before")
others_mask_after = iqr_outlier_mask(others, "pct_after")

assist_summary = factor_summary(assist_clean, pct_col="pct_after")
assist_mask_before = iqr_outlier_mask(assist, "pct_before")
assist_summary_before = factor_summary(assist[~assist_mask_before], pct_col="pct_before")
others_summary_before = factor_summary(others[~others_mask_before], pct_col="pct_before")
others_summary_after = factor_summary(others[~others_mask_after], pct_col="pct_after")

print("Assistance percentage decrease, AFTER-plea base (outliers removed):")
display(assist_summary.round(2))
print()
print("Assistance percentage decrease, BEFORE-plea base (outliers removed):")
display(assist_summary_before.round(2))
print()
print("Other factors — percentage decrease BEFORE plea (base = notional), outliers removed:")
display(others_summary_before.round(2))
print()
print("Other factors — percentage decrease AFTER plea, outliers removed:")
display(others_summary_after.round(2))
print()
print("Outliers removed per factor:")
for label, frame, col in [
	("assistance (after plea)", assist, "pct_after"),
	("other (before plea)", others, "pct_before"),
	("other (after plea)", others, "pct_after"),
]:
	counts = frame.assign(outlier=iqr_outlier_mask(frame, col))
	print(f"-- {label}:")
	print(
		counts.groupby("canonical_factor")["outlier"]
		.agg(["count", "sum"])
		.rename(columns={"count": "charges", "sum": "removed"})
		.to_string()
	)


Assistance percentage decrease, AFTER-plea base (outliers removed):


,factor,charges,mean_pct,median_pct,min_pct,max_pct,pooled_pct
0,Assistance - limited,39,2.86,2.42,0.00,10.53,2.50
1,Assistance - testify,4,29.34,31.08,4.55,50.63,22.24
2,Assistance - useful,17,8.40,5.77,2.48,24.88,7.10
3,Assistance - risk,2,5.11,5.11,3.85,6.38,5.28



Assistance percentage decrease, BEFORE-plea base (outliers removed):


,factor,charges,mean_pct,median_pct,min_pct,max_pct,pooled_pct
0,Assistance - limited,39,2.35,1.82,0.00,7.27,2.50
1,Assistance - testify,4,29.34,31.08,4.55,50.63,22.24
2,Assistance - useful,17,6.49,5.00,1.65,16.77,7.10
3,Assistance - risk,2,4.48,4.48,2.59,6.38,5.28



Other factors — percentage decrease BEFORE plea (base = notional), outliers removed:


,factor,charges,mean_pct,median_pct,min_pct,max_pct,pooled_pct
0,Self-consumption,214,5.80,4.51,0.0,17.65,5.28
1,Rehabilitation programme,51,1.24,1.14,0.0,3.23,1.09
2,Young offender,40,4.16,4.09,0.0,9.52,3.49



Other factors — percentage decrease AFTER plea, outliers removed:


,factor,charges,mean_pct,median_pct,min_pct,max_pct,pooled_pct
0,Self-consumption,210,7.09,6.00,0.0,21.43,5.14
1,Rehabilitation programme,49,1.33,1.21,0.0,3.23,1.04
2,Young offender,40,5.24,5.20,0.0,12.50,3.49



Outliers removed per factor:
-- assistance (after plea):
                      charges  removed
canonical_factor                      
Assistance - limited       42        3
Assistance - risk           2        0
Assistance - testify        4        0
Assistance - useful        19        2
-- other (before plea):
                          charges  removed
canonical_factor                          
Rehabilitation programme       57        6
Self-consumption              220        6
Young offender                 42        2
-- other (after plea):
                          charges  removed
canonical_factor                          
Rehabilitation programme       57        8
Self-consumption              220       10
Young offender                 42        2


In [4]:

GUIDELINE_PLEA = {
	"Guilty plea: Up to committal": 1 / 3,
	"Guilty plea: Plea day": 1 / 3,
	"Guilty plea: After committal": 0.25,
	"Guilty plea: After dates fixed": 0.225,
	"Guilty plea: First day": 0.20,
	"Guilty plea: During trial": 0.15,
	"Guilty plea: Unknown": 1 / 3,
	"Guilty plea: Other": 1 / 3,
}

plea_stage_by_charge = plea_effects.groupby("role_catalogue_key")["canonical_factor"].first()
mitigation["plea_stage"] = mitigation["role_catalogue_key"].map(plea_stage_by_charge)
mitigation["guideline_plea_fraction"] = (
	mitigation["plea_stage"].map(GUIDELINE_PLEA).fillna(0)
)
mitigation["base_after_guideline_plea"] = (
	mitigation["base_months"] * (1 - mitigation["guideline_plea_fraction"])
)

assist_guideline = mitigation[
	mitigation["canonical_factor"].isin(assistance_factors)
	& (mitigation["base_after_guideline_plea"] > 0)
].copy()
assist_guideline["pct_guideline"] = (
	assist_guideline["adjustment_months"] / assist_guideline["base_after_guideline_plea"] * 100
)
assist_guideline_clean = assist_guideline[~iqr_outlier_mask(assist_guideline, "pct_guideline")]

guideline_summary = factor_summary(assist_guideline_clean, pct_col="pct_guideline")
guideline_summary["model_guideline_pct"] = guideline_summary["factor"].map({
	"Assistance - limited": 10.0,
	"Assistance - useful": 15.0,
	"Assistance - testify": 20.0,
	"Assistance - risk": 25.0,
})

assist_comparison = assist_summary[["factor", "median_pct", "mean_pct"]].merge(
	guideline_summary[["factor", "median_pct"]],
	on="factor",
	suffixes=("_actual_plea", "_guideline_plea"),
)
print("Assistance percentage — base after ACTUAL data plea vs GUIDELINE plea, "
      "vs the model guideline:")
display(
	assist_comparison.merge(
		guideline_summary[["factor", "model_guideline_pct"]],
		on="factor",
	).round(2)
)
print("Using the guideline plea reduction leaves the assistance percentages essentially "
      "unchanged (the observed plea reductions are already close to the guideline), so they "
      "stay well below the model guidelines for limited / useful / risk.")


Assistance percentage — base after ACTUAL data plea vs GUIDELINE plea, vs the model guideline:


,factor,median_pct_actual_plea,mean_pct,median_pct_guideline_plea,model_guideline_pct
0,Assistance - limited,2.42,2.86,2.15,10.0
1,Assistance - testify,31.08,29.34,31.08,20.0
2,Assistance - useful,5.77,8.40,5.77,15.0
3,Assistance - risk,5.11,5.11,5.13,25.0


Using the guideline plea reduction leaves the assistance percentages essentially unchanged (the observed plea reductions are already close to the guideline), so they stay well below the model guidelines for limited / useful / risk.


## Fit comparison — before plea vs after plea

For each mitigating factor the reduction is predicted at its outlier-cleaned median
percentage on the chosen base, and the predicted reduction months are compared with the
actual reduction months.  For the **other** factors the error on the before-plea base is
compared with the after-plea base to decide which ordering fits better.


In [5]:

def evaluate(frame, base_col, predicted_by_factor):
	rows = []
	for factor, group in frame.groupby("canonical_factor", sort=False):
		fraction = predicted_by_factor[factor]
		predicted_months = group[base_col] * fraction
		error_months = predicted_months - group["adjustment_months"]
		positive = group["adjustment_months"] > 0
		rel = (error_months.abs() / group["adjustment_months"])[positive]
		rows.append({
			"factor": factor,
			"charges": len(group),
			"predicted_pct": round(fraction * 100, 2),
			"mae_months": round(error_months.abs().mean(), 2),
			"median_abs_error_months": round(error_months.abs().median(), 2),
			"mape_pct": round(rel.mean() * 100, 1),
			"within_25pct_pct": round((rel <= 0.25).mean() * 100, 1),
			"within_50pct_pct": round((rel <= 0.50).mean() * 100, 1),
		})
	return pd.DataFrame(rows)


assist_pred = assist_summary.set_index("factor")["median_pct"] / 100
assist_error = evaluate(assist, "base_after_plea", assist_pred)
print("Assistance — error using after-plea base (all charges):")
display(assist_error)

others_pred_before = others_summary_before.set_index("factor")["median_pct"] / 100
others_pred_after = others_summary_after.set_index("factor")["median_pct"] / 100
others_error_before = evaluate(others, "base_months", others_pred_before)
others_error_after = evaluate(others, "base_after_plea", others_pred_after)

comparison = others_error_before.merge(
	others_error_after,
	on="factor",
	suffixes=("_before", "_after"),
)
print("Other factors — before vs after plea (lower MAE / higher hit rate wins):")
display(
	comparison[["factor", "charges_before",
				"predicted_pct_before", "predicted_pct_after",
				"mae_months_before", "mae_months_after",
				"median_abs_error_months_before", "median_abs_error_months_after",
				"within_25pct_pct_before", "within_25pct_pct_after",
				"within_50pct_pct_before", "within_50pct_pct_after"]]
)


Assistance — error using after-plea base (all charges):


,factor,charges,predicted_pct,mae_months,median_abs_error_months,mape_pct,within_25pct_pct,within_50pct_pct
0,Assistance - limited,42,2.42,6.98,3.04,70.2,17.6,38.2
1,Assistance - testify,4,31.08,29.85,27.13,206.9,0.0,50.0
2,Assistance - useful,19,5.77,10.07,5.93,55.0,31.6,47.4
3,Assistance - risk,2,5.11,2.28,2.28,26.4,50.0,100.0


Other factors — before vs after plea (lower MAE / higher hit rate wins):


,factor,charges_before,predicted_pct_before,predicted_pct_after,mae_months_before,mae_months_after,median_abs_error_months_before,median_abs_error_months_after,within_25pct_pct_before,within_25pct_pct_after,within_50pct_pct_before,within_50pct_pct_after
0,Self-consumption,220,4.51,6.00,2.59,2.63,1.79,1.84,23.8,23.3,51.5,52.5
1,Rehabilitation programme,57,1.14,1.21,1.22,1.33,0.91,1.15,35.8,26.4,66.0,60.4
2,Young offender,42,4.09,5.20,3.09,3.33,1.31,1.61,31.6,28.9,65.8,60.5


In [6]:

def test_set(frame, pct_col, per_factor=3):
	chunks = []
	for factor in all_mitigating_factors:
		group = frame[frame["canonical_factor"] == factor]
		if group.empty:
			continue
		chunks.append(group.sort_values("case_id").head(per_factor).assign(
			canonical_factor=factor, pct=pct_col))
	columns = [
		"canonical_factor",
		"neutral_citation",
		"trial_index",
		"charge_no",
		"defendant_id",
		"base_months",
		"base_after_plea",
		"plea_reduction_months",
		"adjustment_months",
		"pct",
	]
	if not chunks:
		return pd.DataFrame(columns=columns)
	result = pd.concat(chunks, ignore_index=True)
	result["pct"] = result[pct_col]
	return result[columns]

test = test_set(mitigation, "pct_after")
print("Small test set per factor, AFTER-plea percentages (for manual verification):")
print(test.round(2).to_string(index=False))


Small test set per factor, AFTER-plea percentages (for manual verification):
        canonical_factor  neutral_citation  trial_index  charge_no  defendant_id  base_months  base_after_plea  plea_reduction_months  adjustment_months   pct
    Assistance - limited [2021] HKCFI 1694            0          1             1        261.0           261.00                   0.00               3.00  1.15
    Assistance - limited [2021] HKCFI 2372            0          1             1        276.0           276.00                   0.00               0.00  0.00
    Assistance - limited [2021] HKCFI 2598            0          1             1        300.0           300.00                   0.00              21.00  7.00
     Assistance - useful [2021] HKCFI 2702            0          1             1        159.0           107.01                  51.99               3.00  2.80
     Assistance - useful  [2021] HKCFI 968            0          2             1        252.0           252.00                  

In [7]:

report_path = notebook_dir / "mitigating_factor_adjustments_analysis.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
	plea_report.round(2).to_excel(writer, sheet_name="guilty plea", index=False)
	assist_summary.round(2).to_excel(writer, sheet_name="assistance (after plea)", index=False)
	assist_summary_before.round(2).to_excel(writer, sheet_name="assistance (before plea)", index=False)
	guideline_summary.round(2).to_excel(writer, sheet_name="assistance (guideline plea)", index=False)
	others_summary_before.round(2).to_excel(writer, sheet_name="other factors (before plea)", index=False)
	others_summary_after.round(2).to_excel(writer, sheet_name="other factors (after plea)", index=False)
	assist_error.to_excel(writer, sheet_name="assistance error", index=False)
	comparison.round(2).to_excel(writer, sheet_name="fit before vs after", index=False)
	test.round(2).to_excel(writer, sheet_name="test set", index=False)
print("Wrote", report_path)


Wrote /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/predictionModel/mitigating_factor_adjustments_analysis.xlsx


## Reading the results

- Guilty plea percentages are relative to the **pre-plea** sentence; the medians should be
  compared against the model defaults (33.3 / 25 / 22.5 / 20 / 15%).
- Assistance is measured on the **after-plea** base (`notional - plea reduction`).
- For Self-consumption, Young offender and Rehabilitation programme the `fit before vs
  after` sheet shows which base gives the lower error.
- Small groups (Assistance - testify with 4 charges, Assistance - risk with 2) are
  illustrative only.
